# Lesson 000.24 -- The Canonical Envelope: Local JSON, SHA-256, and the Trust Boundary

AuthorityEffect: None.

> **Claim-typing legend**
> - **[Pinned]**: reserved for a cited checked-in specification; this illustrative class is not the production packet schema.
> - **[Executed]**: a finite code-check label, not a learner outcome or general conformance proof; a run needs its own evidence.
> - **[Declared]**: pedagogical convention or beginner framing.
> - **[Open]**: questions left for future exploration.

## 1. The 4-Field PacketEnvelope Kernel

[Declared] This worked example uses an illustrative four-field immutable record, not the complete production packet schema:
```python
@dataclass(frozen=True)
class PacketEnvelope:
    carrier: str         # e.g., 'MSS-1', 'MSS-000-CAPSTONE'
    input_state: str     # State before candidate transition
    candidate_move: str  # Proposed move or cut
    host_verdict: str    # Referee verdict ('ADMITTED', 'REFUSED_...')
```


In [ ]:
from dataclasses import asdict, dataclass


@dataclass(frozen=True)
class PacketEnvelope:
    carrier: str
    input_state: str
    candidate_move: str
    host_verdict: str


# Instantiate immutable envelope
env = PacketEnvelope(
    carrier="MSS-1",
    input_state="c=(2, -1, 0)",
    candidate_move="phi=(0, -1, 0)",
    host_verdict="ADMITTED",
)

assert env.carrier == "MSS-1"
assert env.input_state == "c=(2, -1, 0)"
assert env.candidate_move == "phi=(0, -1, 0)"
assert env.host_verdict == "ADMITTED"

print("Illustrative four-field fixture:", env)

## 2. Local Python JSON Profile and Digest Consistency vs. Host Authority

[Declared] Local serialization rules for the four-string-field fixture; not general standards conformance:
1. Python sorts the four ASCII field names with `sort_keys=True`; general Unicode-key and cross-language ordering are not tested here.
2. Zero whitespace outside string literals: `separators=(',', ':')`.
3. Cryptographic commitment: `SHA-256(canonical_envelope_json)`.

### Crucial Epistemic Boundary: Matching Digest != Host Authorization
Testing `compute_digest(env) == declared_receipt` verifies only data consistency against a receipt.
An adversary can forge an envelope with `host_verdict='ADMITTED'` and compute its fresh digest; that self-check passes!
Establishing genuine host origin requires:
1. **Schema Validation:** Enforcing string types, registered carriers, and closed verdict vocabulary.
2. **Trusted Commitment Matching:** Comparing against an independently authenticated, trusted host commitment. This example constructs a local fixture commitment and assumes trust; it does not authenticate a host or verify a signature.


In [ ]:
import hashlib
import json
from dataclasses import dataclass

ALLOWED_CARRIERS = {"MSS-1", "MSS-000-CAPSTONE"}
ALLOWED_VERDICTS = {
    "ADMITTED",
    "REFUSED_DEFECT_INCREASE",
    "REFUSED_SYNTAX",
    "REFUSED_DOMAIN",
    "REFUSED_CARRIER",
    "REFUSED_COHOMOLOGY_BREACH",
}


def canonical_envelope_json(envelope: PacketEnvelope) -> str:
    """Local sorted-compact Python JSON for the illustrated string-field record."""
    data = asdict(envelope)
    for k, v in data.items():
        if not isinstance(k, str) or not isinstance(v, str):
            raise TypeError(f"Envelope field {k} must have string value, got {type(v)}")
    return json.dumps(data, sort_keys=True, separators=(",", ":"), ensure_ascii=False)


def compute_digest(envelope: PacketEnvelope) -> str:
    serialized = canonical_envelope_json(envelope)
    return hashlib.sha256(serialized.encode("utf-8")).hexdigest()


def validate_envelope_schema(envelope: PacketEnvelope) -> None:
    if not isinstance(envelope, PacketEnvelope):
        raise TypeError("Object is not a PacketEnvelope")
    for f in ("carrier", "input_state", "candidate_move", "host_verdict"):
        val = getattr(envelope, f, None)
        if not isinstance(val, str):
            raise TypeError(f"Field '{f}' must be str, got {type(val)}")
    if envelope.carrier not in ALLOWED_CARRIERS:
        raise ValueError(f"Unrecognized carrier '{envelope.carrier}'")
    if envelope.host_verdict not in ALLOWED_VERDICTS:
        raise ValueError(f"Unrecognized host_verdict '{envelope.host_verdict}'")


def verify_envelope_against_commitment(envelope: PacketEnvelope, trusted_commitment: str) -> bool:
    try:
        validate_envelope_schema(envelope)
    except (TypeError, ValueError):
        return False
    return compute_digest(envelope) == trusted_commitment


# 1. Locally constructed genuine fixture; authentication is assumed
env_genuine = PacketEnvelope(
    carrier="MSS-1",
    input_state="c=(2, -1, 0)",
    candidate_move="phi=(0, -1, 0)",
    host_verdict="ADMITTED",
)
trusted_host_commitment = compute_digest(env_genuine)

# 2. Genuine envelope passes verification against trusted commitment
assert verify_envelope_against_commitment(env_genuine, trusted_host_commitment)

# 3. Forged envelope: adversary writes ADMITTED and calculates its digest
forged_env = PacketEnvelope(
    carrier="MSS-1",
    input_state="c=(2, -1, 0)",
    candidate_move="c=(2, 2, 2)",
    host_verdict="ADMITTED",
)
forged_digest = compute_digest(forged_env)

# Forged envelope fails when checked against trusted host commitment!
assert not verify_envelope_against_commitment(forged_env, trusted_host_commitment)

# 4. Schema validation prevents forged envelopes with invalid verdicts or non-string fields
invalid_verdict_env = PacketEnvelope(
    carrier="MSS-1",
    input_state="c=(2, -1, 0)",
    candidate_move="c=(2, 2, 2)",
    host_verdict="WEAPONIZED_BYPASS",
)
assert not verify_envelope_against_commitment(
    invalid_verdict_env, compute_digest(invalid_verdict_env)
)

print("Genuine digest: ", trusted_host_commitment[:16] + "...")
print("Forger digest:  ", forged_digest[:16] + "...")
print("Local fixture schema and commitment-comparison checks passed.")

## 3. Diagnostic Probe (2026 Micro-Attention Check)

**Question:** If an external proposer submits a `PacketEnvelope` whose calculated SHA-256 digest matches its accompanying receipt string, why is that check alone insufficient to certify that the Host referee admitted the move?

- **A:** Because SHA-256 has too many hash collisions.
- **B:** Because anyone can alter the fields of an untrusted envelope (such as writing `host_verdict = 'ADMITTED'`) and freshly recompute its SHA-256 digest; authorization requires checking schema validity and matching against an independently authenticated, trusted host commitment.
- **C:** Because JSON strings cannot be hashed.

*(Correct answer: **B**)*


In [ ]:
def diagnostic_probe(choice: str) -> bool:
    return choice.strip().upper() == "B"


assert diagnostic_probe("B")
assert not diagnostic_probe("A")
assert not diagnostic_probe("C")

# Student Exercise: Verify that tampered envelope is rejected by trusted commitment
tampered_env = PacketEnvelope(
    carrier="MSS-1",
    input_state="c=(2, -1, 0)",
    candidate_move="phi=(0, -1, 0)",
    host_verdict="REFUSED_DEFECT_INCREASE",
)
assert not verify_envelope_against_commitment(tampered_env, trusted_host_commitment)

print("Lesson 000.24 diagnostic probe and validator exercise verified successfully!")